# 식물 카드 → 벡터 DB 만들기 + 질문 5개 테스트

**준비**: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 (임베딩이 훨씬 빨라요). API 키는 필요 없어요.

순서: ① 설치 → ② 카드 zip 올리기 → ③ 카드 읽고 조각 내기 → ④ 벡터 DB 만들기 → ⑤ 질문 5개 테스트 → ⑥ DB 저장


## ① 설치 (1～2분)

In [ ]:
!pip install -q langchain-chroma langchain-huggingface sentence-transformers

## ② 카드 zip 올리기
실행하면 파일 선택 버튼이 나와요. `cards_20261116.zip`을 고르세요.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zname = list(up)[0]
zipfile.ZipFile(zname).extractall("/content")
print("카드 폴더:", os.listdir("/content/cards")[:5], "...")

## ③ 카드 읽고 조각 내기
카드는 `## 빛`, `## 물주기`처럼 주제별로 나뉘어 있어서, **주제(섹션) 하나 = 조각 하나**로 잘라요. 조각마다 `[식물 이름] 주제`를 앞에 붙이고, 식물 이름·그룹(30종/그외)·출처를 메타데이터로 달아요.

In [ ]:
# 카드 읽기 → 섹션별 조각(Document) 만들기
import re
from pathlib import Path

def parse_card(path):
    text = Path(path).read_text(encoding="utf-8")
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.S)
    meta, body = {}, text
    if m:
        for line in m.group(1).splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.split("#")[0].strip() if k.strip() == "group" else v.strip()
        body = m.group(2)
    return meta, body

def split_sections(body):
    """'## 제목' 단위로 자르기. 카드가 이미 주제별로 나뉘어 있어서 글자 수로 자르는 것보다 정확해요."""
    parts, title, buf = [], "개요", []
    for line in body.splitlines():
        if line.startswith("## "):
            if "".join(buf).strip():
                parts.append((title, "\n".join(buf).strip()))
            title, buf = line[3:].strip(), []
        elif line.startswith("# "):
            continue
        else:
            buf.append(line)
    if "".join(buf).strip():
        parts.append((title, "\n".join(buf).strip()))
    return parts

def card_chunks(card_dir):
    chunks = []
    for p in sorted(Path(card_dir).rglob("*.md")):
        if p.name.startswith("_"):
            continue  # 양식 파일 제외
        meta, body = parse_card(p)
        name = meta.get("plant_ko", p.stem)
        for sec, txt in split_sections(body):
            content = f"[{name}] {sec}\n{txt}"  # 조각마다 식물 이름을 붙여서 검색이 잘 되게
            md = {k: meta.get(k, "") for k in ("plant_id", "plant_ko", "group", "category", "source", "source_type", "url", "aliases", "scientific_name")}
            md.update({"section": sec, "file": p.name})
            chunks.append((content, md))
    return chunks

GENERAL = {"베란다", "식물", "빛", "햇빛", "북향", "남향", "동향", "서향", "겨울", "여름", "초보", "초보자", "물", "화분", "흙", "추천", "창가", "식물등", "그늘"}
JOSA = r"(은|는|이|가|을|를|도|의|에|에서|랑|하고|와|과|로|으로)$"

def plant_names(chunks):
    names = {}
    for _, md in chunks:
        ko = md["plant_ko"]
        names[ko] = ko
        for a in re.split(r"[,(/]", md.get("aliases", "")):
            a = re.sub(r"추천 유통명:|\)", "", a).strip()
            if len(a) >= 2:
                names.setdefault(a, ko)
    return names

def find_plant(query, names):
    """질문에서 식물 이름 찾기. 반환: (식물이름 or None, 첫 단어가 모르는 식물이면 그 단어)"""
    hits = [n for n in names if n in query]
    if hits:
        return names[max(hits, key=len)], None
    first = re.sub(JOSA, "", query.split()[0]) if query.split() else ""
    if first and first not in GENERAL and not any(g in first for g in GENERAL):
        return None, first
    return None, None


chunks = card_chunks("/content/cards")
NAMES = plant_names(chunks)
import collections
print("조각 수:", len(chunks))
print("그룹별:", collections.Counter(m["group"] for _, m in chunks))
print("식물 이름(별명 포함):", len(NAMES))
print("\n예시 조각:\n", chunks[0][0][:200])

## ④ 벡터 DB 만들기 (GPU 약 3～5분)
무료 한국어 임베딩 모델 **BAAI/bge-m3**를 써요. 처음에 모델(약 2GB)을 내려받아요.

In [ ]:
import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용 장치:", device, "(cpu면 20분 이상 걸릴 수 있어요)")
emb = HuggingFaceEmbeddings(model_name="BAAI/bge-m3",
                            model_kwargs={"device": device},
                            encode_kwargs={"normalize_embeddings": True})
docs = [Document(page_content=t, metadata=m) for t, m in chunks]
db = Chroma.from_documents(docs, emb, collection_name="plants",
                           persist_directory="/content/plant_db",
                           collection_metadata={"hnsw:space": "cosine"})
print("DB에 들어간 조각 수:", db._collection.count())

## ⑤ 질문 5개 테스트
검색 규칙 (나중에 LLM이 답을 쓸 때도 이 규칙 그대로 써요):
- 질문에 **DB에 있는 식물 이름(별명 포함)**이 있으면 → 그 식물 카드에서만 찾음
- 질문 첫 단어가 **DB에 없는 식물 이름**이면 → "자료 없음" (지어내지 않음)
- 식물 이름이 없는 일반 질문(예: 북향 베란다 추천)이면 → 전체에서 찾음

점수(distance)는 **작을수록 질문과 가까운** 조각이에요.

In [ ]:
def search(query, k=3):
    plant, unknown = find_plant(query, NAMES)
    print("=" * 70)
    print("질문:", query)
    if unknown:
        print(f"→ '{unknown}'은(는) DB에 없는 식물이에요 → 자료 없음 (답을 지어내지 않음)")
        return []
    flt = {"plant_ko": plant} if plant else None
    print("→ 찾은 식물:", plant or "(없음: 전체에서 검색)")
    res = db.similarity_search_with_score(query, k=k, filter=flt)
    for d, s in res:
        m = d.metadata
        print(f"  [{s:.3f}] {m['plant_ko']} / {m['section']} / {m['group']} / {m['source_type']}: {m['source'][:40]}")
        print("     ", d.page_content.split("\n", 1)[-1][:120].replace("\n", " "))
    return res

TESTS = [
    "상추 베란다에서 키울 때 물은 얼마나 줘?",          # 30종 (책 자료)
    "자미오쿨카스 겨울에 베란다에 둬도 돼?",            # 별명 → 금전수 (농사로 API)
    "빛이 부족한 북향 베란다에서 키울 수 있는 식물 추천해줘",  # 일반 질문 → 전체 검색
    "블루베리 화분 흙은 뭘 써야 해?",                 # 그외 과일 (책 자료)
    "수국 키우는 법 알려줘",                          # DB에 없음 → 자료 없음
]
for q in TESTS:
    search(q)

직접 질문을 바꿔서 더 해 봐도 돼요. 예: `search("홍콩야자 잎이 처져요")`

In [ ]:
search("홍콩야자 잎이 처져요")

## ⑥ DB 저장
만든 DB를 zip으로 내려받아요. 다음에 LLM을 붙이거나 앱 서버에 올릴 때 이 파일을 그대로 써요 (다시 만들 필요 없음).

In [ ]:
import shutil
shutil.make_archive("/content/plant_db", "zip", "/content/plant_db")
files.download("/content/plant_db.zip")